In [73]:
import random

def first_choice_hc(landscape, start):
    current = start
    path = [current]

    while True:
        moved = False
        current_value = landscape[current - 1]

        if current > 1:
            left = current - 1
            if landscape[left - 1] > current_value:
                current = left
                path.append(current)
                moved = True

        if not moved and current < len(landscape):
            right = current + 1
            if landscape[right - 1] > current_value:
                current = right
                path.append(current)
                moved = True

        if not moved:
            break

    return path, current


def stochastic_hc(landscape, start):
    current = start
    path = [current]

    while True:
        current_value = landscape[current - 1]
        neighbors = []

        if current > 1:
            left = current - 1
            if landscape[left - 1] > current_value:
                neighbors.append(left)

        if current < len(landscape):
            right = current + 1
            if landscape[right - 1] > current_value:
                neighbors.append(right)

        if not neighbors:
            break

        current = random.choice(neighbors)
        path.append(current)

    return path, current


def main():
    landscape = [4, 9, 6, 11, 8, 15, 10, 7, 13, 5, 16, 12]

    print(f"{'Start':<6}{'Algo':<12}{'Path':<30}{'Terminal':<10}{'Steps'}")
    print("-" * 70)

    for s in range(1, 13):
        path, term = first_choice_hc(landscape, s)
        print(f"{s:<6}{'FirstChoice':<12}{str(path):<30}{term:<10}{len(path)-1}")

        path, term = stochastic_hc(landscape, s)
        print(f"{s:<6}{'Stochastic':<12}{str(path):<30}{term:<10}{len(path)-1}")

main()

Start Algo        Path                          Terminal  Steps
----------------------------------------------------------------------
1     FirstChoice [1, 2]                        2         1
1     Stochastic  [1, 2]                        2         1
2     FirstChoice [2]                           2         0
2     Stochastic  [2]                           2         0
3     FirstChoice [3, 2]                        2         1
3     Stochastic  [3, 4]                        4         1
4     FirstChoice [4]                           4         0
4     Stochastic  [4]                           4         0
5     FirstChoice [5, 4]                        4         1
5     Stochastic  [5, 6]                        6         1
6     FirstChoice [6]                           6         0
6     Stochastic  [6]                           6         0
7     FirstChoice [7, 6]                        6         1
7     Stochastic  [7, 6]                        6         1
8     FirstChoice [8, 7, 

In [74]:
def experiment():
    landscape = [4, 9, 6, 11, 8, 15, 10, 7, 13, 5, 16, 12]

    success = 0

    for _ in range(50):
        _, term = stochastic_hc(landscape, 4)
        if term == 11:
            success += 1
    print("\n50 Run Experiment:")
    print("Reached 11:", success)
    print("Local max:", 50 - success)
    print("Success %:", (success/50)*100)

experiment()


50 Run Experiment:
Reached 11: 0
Local max: 50
Success %: 0.0


In [75]:
def first_choice_plateau(landscape, start):
    current = start
    path = [current]

    while True:
        moved = False
        current_value = landscape[current - 1]

        equal_found = False

        if current > 1:
            left = current - 1
            val = landscape[left - 1]

            if val > current_value:
                current = left
                path.append(current)
                moved = True
            elif val == current_value:
                equal_found = True

        if not moved and current < len(landscape):
            right = current + 1
            val = landscape[right - 1]

            if val > current_value:
                current = right
                path.append(current)
                moved = True
            elif val == current_value:
                equal_found = True

        if not moved:
            if equal_found:
                print(f"Plateau detected at state {current}")
            break

    return path, current

def stochastic_plateau(landscape, start):
    current = start
    path = [current]

    while True:
        current_value = landscape[current - 1]

        better = []
        equal = []

        if current > 1:
            left = current - 1
            val = landscape[left - 1]
            if val > current_value:
                better.append(left)
            elif val == current_value:
                equal.append(left)

        if current < len(landscape):
            right = current + 1
            val = landscape[right - 1]
            if val > current_value:
                better.append(right)
            elif val == current_value:
                equal.append(right)

        if better:
            current = random.choice(better)
            path.append(current)
        else:
            if equal:
                print(f"Plateau detected at state {current}")
            break

    return path, current

In [76]:
def plateau_experiment_before():
    landscape = [4, 9, 6, 11, 15, 15, 15, 7, 13, 5, 16, 12]

    fc_plateau = 0
    st_plateau = 0
    fc_success = 0
    st_success = 0

    for s in range(1, 13):
        _, t1 = first_choice_plateau(landscape, s)
        _, t2 = stochastic_plateau(landscape, s)

        if t1 in [5,6,7]:
            fc_plateau += 1
        if t2 in [5,6,7]:
            st_plateau += 1

        if t1 == 11:
            fc_success += 1
        if t2 == 11:
            st_success += 1

    print("\n--- BEFORE SIDEWAYS ---")
    print("FC plateau:", fc_plateau)
    print("ST plateau:", st_plateau)
    print("FC success:", fc_success)
    print("ST success:", st_success)

In [77]:
def first_choice_sideways(landscape, start, limit=10):
    current = start
    path = [current]
    sideways = 0
    plateau_flag = False

    while True:
        moved = False
        current_value = landscape[current - 1]
        equal = []

        if current > 1:
            left = current - 1
            val = landscape[left - 1]
            if val > current_value:
                current = left
                path.append(current)
                sideways = 0
                plateau_flag = False
                moved = True
            elif val == current_value:
                equal.append(left)

        if not moved and current < len(landscape):
            right = current + 1
            val = landscape[right - 1]
            if val > current_value:
                current = right
                path.append(current)
                sideways = 0
                plateau_flag = False
                moved = True
            elif val == current_value:
                equal.append(right)

        if not moved:
            if equal:
                if not plateau_flag:
                    print(f"Plateau detected at state {current}")
                    plateau_flag = True

                if sideways < limit:
                    current = equal[0]
                    path.append(current)
                    sideways += 1
                    print(f"Sideways -> {current}")
                else:
                    print(f"Sideways limit reached at state {current}")
                    break
            else:
                break

    return path, current

def stochastic_hc_sideways(landscape, start, sideways_limit=10):
    current = start
    path = [current]
    sideways_moves = 0
    plateau_flag = False

    while True:
        current_value = landscape[current - 1]

        better_neighbors = []
        equal_neighbors = []

        if current > 1:
            left = current - 1
            val = landscape[left - 1]
            if val > current_value:
                better_neighbors.append(left)
            elif val == current_value:
                equal_neighbors.append(left)

        if current < len(landscape):
            right = current + 1
            val = landscape[right - 1]
            if val > current_value:
                better_neighbors.append(right)
            elif val == current_value:
                equal_neighbors.append(right)

        if better_neighbors:
            current = random.choice(better_neighbors)
            path.append(current)
            sideways_moves = 0
            plateau_flag = False
            continue

        if equal_neighbors:
            if not plateau_flag:
                print(f"Plateau detected at state {current}")
                plateau_flag = True

            if sideways_moves < sideways_limit:
                current = random.choice(equal_neighbors)
                path.append(current)
                sideways_moves += 1
                print(f"Sideways -> {current}")
                continue
            else:
                print(f"Sideways limit reached at state {current}")
                break

        break

    return path, current

In [78]:
def plateau_experiment_after():
    landscape = [4, 9, 6, 11, 15, 15, 15, 7, 13, 5, 16, 12]

    fc_plateau = 0
    st_plateau = 0
    fc_success = 0
    st_success = 0

    for s in range(1, 13):
        _, t1 = first_choice_sideways(landscape, s)
        _, t2 = stochastic_hc_sideways(landscape, s, sideways_limit=10)

        if t1 in [5,6,7]:
            fc_plateau += 1
        if t2 in [5,6,7]:
            st_plateau += 1

        if t1 == 11:
            fc_success += 1
        if t2 == 11:
            st_success += 1

    print("\n--- AFTER SIDEWAYS ---")
    print("FC plateau:", fc_plateau)
    print("ST plateau:", st_plateau)
    print("FC success:", fc_success)
    print("ST success:", st_success)

In [79]:
def main():
    print("Running BEFORE sideways...")
    plateau_experiment_before()

    print("\nRunning AFTER sideways...")
    plateau_experiment_after()

main()

Running BEFORE sideways...
Plateau detected at state 5
Plateau detected at state 5
Plateau detected at state 5
Plateau detected at state 5
Plateau detected at state 5
Plateau detected at state 6
Plateau detected at state 6
Plateau detected at state 7
Plateau detected at state 7
Plateau detected at state 7
Plateau detected at state 7

--- BEFORE SIDEWAYS ---
FC plateau: 5
ST plateau: 6
FC success: 2
ST success: 2

Running AFTER sideways...
Plateau detected at state 5
Sideways -> 6
Sideways -> 5
Sideways -> 6
Sideways -> 5
Sideways -> 6
Sideways -> 5
Sideways -> 6
Sideways -> 5
Sideways -> 6
Sideways -> 5
Sideways limit reached at state 5
Plateau detected at state 5
Sideways -> 6
Sideways -> 7
Sideways -> 6
Sideways -> 7
Sideways -> 6
Sideways -> 7
Sideways -> 6
Sideways -> 5
Sideways -> 6
Sideways -> 5
Sideways limit reached at state 5
Plateau detected at state 5
Sideways -> 6
Sideways -> 5
Sideways -> 6
Sideways -> 5
Sideways -> 6
Sideways -> 5
Sideways -> 6
Sideways -> 5
Sideways -> 6